# In this notebook, we will explore the use of a Graph-VAE for our task.
We will build off of the existing model based on the Gómez-Bombarelli / `molecular-vae` style (conv encoder + GRU decoder).

## Imports

In [ ]:
from pathlib import Path
import random
from dataclasses import dataclass
from typing import Any, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import selfies as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
MAX_LEN = 120  # maximum SELFIES token length before we append EOS
VAL_FRAC = 0.10
TEST_FRAC = 0.10

# Keep the latent size and decoder dimensions aligned with the old VAE.
# That way the main architectural change is the graph encoder.
LATENT_DIM = 292
GRAPH_HIDDEN_DIM = 292
GRAPH_BLOCKS = 3
GRAPH_READOUT_DIM = 435
DECODER_INPUT_DIM = 292
DECODER_HIDDEN_DIM = 501
DECODER_LAYERS = 3

EPOCHS = 20
BATCH_SIZE = 128
LR = 1e-3
KL_ANNEAL_EPOCHS = 10
FREE_BITS_NATS = 0.0
RUN_TRAINING = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
)
print('device:', device)
print('torch:', torch.__version__)
print('selfies:', sf.__version__)


In [ ]:
DATA_ROOT = Path('data')
CHEMBL_PATH = DATA_ROOT / 'Train' / 'chembl_clean.csv'
ZINC_PATH = DATA_ROOT / 'Train' / 'zinc250k_clean.csv'
TOX21_TRAIN_PATH = DATA_ROOT / 'Train' / 'tox21_train_clean.csv'
TOX21_VAL_PATH = DATA_ROOT / 'Val' / 'tox21_val_clean.csv'
TOX21_TEST_PATH = DATA_ROOT / 'Test' / 'tox21_test_clean.csv'

for p in [CHEMBL_PATH, ZINC_PATH, TOX21_TRAIN_PATH, TOX21_VAL_PATH, TOX21_TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing file: {p}')


def load_smiles(path: Path) -> list[str]:
    df = pd.read_csv(path)
    if 'canonical_smiles' not in df.columns:
        raise ValueError(f'{path} does not contain canonical_smiles')
    smiles = df['canonical_smiles'].dropna().astype(str).tolist()
    return list(dict.fromkeys(smiles))


chembl_smiles = load_smiles(CHEMBL_PATH)
zinc_smiles = load_smiles(ZINC_PATH)
pretrain_smiles = list(dict.fromkeys(chembl_smiles + zinc_smiles))

# We still load tox21 here because it is useful for later downstream evaluation,
# even though the notebook below focuses on graph-VAE pretraining.
tox21_train_smiles = load_smiles(TOX21_TRAIN_PATH)
tox21_val_smiles = load_smiles(TOX21_VAL_PATH)
tox21_test_smiles = load_smiles(TOX21_TEST_PATH)

print(f'ChemBL unique SMILES: {len(chembl_smiles):,}')
print(f'Zinc unique SMILES:   {len(zinc_smiles):,}')
print(f'Combined pretrain pool: {len(pretrain_smiles):,}')
print(f'tox21 train/val/test: {len(tox21_train_smiles):,} / {len(tox21_val_smiles):,} / {len(tox21_test_smiles):,}')


## Prepare Paired Graph Inputs And SELFIES Targets
We want every training example to contain two synchronized views of the same molecule:
- a **graph** view for the encoder
- a **SELFIES sequence** view for the decoder target


In [ ]:
PAD = '<PAD>'
UNK = '<UNK>'
EOS = '<EOS>'


def smiles_to_filtered_selfies(smiles_list: list[str], max_len: int = MAX_LEN):
    """
    Convert SMILES to SELFIES and drop molecules that the decoder would not be able to handle.

    We keep the SMILES string and the SELFIES string together so the graph encoder
    and the sequence decoder always refer to the same molecule.
    """
    kept_smiles = []
    kept_selfies = []
    failed = 0
    too_long = 0

    for smi in smiles_list:
        try:
            selfies_str = sf.encoder(smi)
        except Exception:
            failed += 1
            continue

        token_count = len(list(sf.split_selfies(selfies_str)))
        if token_count > max_len:
            too_long += 1
            continue

        kept_smiles.append(smi)
        kept_selfies.append(selfies_str)

    return kept_smiles, kept_selfies, failed, too_long


def tokenize_selfies(selfies_str: str) -> list[str]:
    return list(sf.split_selfies(selfies_str))


filtered_smiles, filtered_selfies, pretrain_failed, pretrain_too_long = smiles_to_filtered_selfies(pretrain_smiles)

train_smiles, temp_smiles, train_selfies, temp_selfies = train_test_split(
    filtered_smiles,
    filtered_selfies,
    test_size=VAL_FRAC + TEST_FRAC,
    random_state=SEED,
    shuffle=True,
)

val_ratio_in_temp = VAL_FRAC / (VAL_FRAC + TEST_FRAC)
val_smiles, test_smiles, val_selfies, test_selfies = train_test_split(
    temp_smiles,
    temp_selfies,
    test_size=(1 - val_ratio_in_temp),
    random_state=SEED,
    shuffle=True,
)

train_tokens = [tokenize_selfies(s) for s in train_selfies]
vocab_tokens = sorted({tok for seq in train_tokens for tok in seq})
ALL_TOKENS = [PAD, UNK, EOS] + vocab_tokens
TOKEN_TO_IDX = {tok: i for i, tok in enumerate(ALL_TOKENS)}
IDX_TO_TOKEN = {i: tok for tok, i in TOKEN_TO_IDX.items()}

PAD_IDX = TOKEN_TO_IDX[PAD]
UNK_IDX = TOKEN_TO_IDX[UNK]
EOS_IDX = TOKEN_TO_IDX[EOS]
SEQ_LEN = MAX_LEN + 1
VOCAB_SIZE = len(ALL_TOKENS)


def encode_selfies(selfies_str: str) -> list[int]:
    ids = [TOKEN_TO_IDX.get(tok, UNK_IDX) for tok in tokenize_selfies(selfies_str)]
    ids = ids[:MAX_LEN]
    ids.append(EOS_IDX)
    return ids


def encode_list(selfies_list: list[str]) -> np.ndarray:
    out = np.full((len(selfies_list), SEQ_LEN), PAD_IDX, dtype=np.int64)
    for i, selfies_str in enumerate(selfies_list):
        ids = encode_selfies(selfies_str)
        out[i, :len(ids)] = ids
    return out


train_targets = encode_list(train_selfies)
val_targets = encode_list(val_selfies)
test_targets = encode_list(test_selfies)

print(f'SELFIES conversion failures: {pretrain_failed}')
print(f'SELFIES dropped for length > {MAX_LEN}: {pretrain_too_long}')
print(f'filtered pretrain molecules: {len(filtered_smiles):,}')
print(f'split sizes: train={len(train_smiles):,}, val={len(val_smiles):,}, test={len(test_smiles):,}')
print(f'VOCAB_SIZE={VOCAB_SIZE}, SEQ_LEN={SEQ_LEN}')
print('target array shapes:', train_targets.shape, val_targets.shape, test_targets.shape)


## Graph Featurizer
For a toxicity-oriented molecular encoder, **atom type + bond type** is a good starting point, but it leaves out several local chemistry signals that often matter for biological activity and toxicity.

In this notebook we enrich the fixed graph inputs with a small set of **cheap, local, RDKit-derived features** that are still easy to reason about:

**Node features**
- atom type
- formal charge
- degree
- hybridization
- total attached hydrogens
- aromatic flag
- ring-membership flag

**Edge features**
- bond type
- bond stereo
- conjugated flag
- ring-membership flag

The most important conceptual point is this: **RDKit computes these chemistry features once per molecule at featurization time.** The Graph Nets layers do *not* call RDKit again. They only operate on learned hidden states built from these fixed inputs.

That means adding a few local chemistry features is usually a very good tradeoff: you give the model better inductive bias without making the message-passing architecture itself more complicated.


In [ ]:
# Build the atom vocabulary from the training split only.
# This mirrors how we build the SELFIES vocabulary for the sequence model:
# the training data defines the feature space, and we keep one fallback 'Unknown' bucket.
def build_atom_vocab(smiles_list: list[str]) -> list[str]:
    atom_symbols = set()
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue
        for atom in mol.GetAtoms():
            atom_symbols.add(atom.GetSymbol())
    return sorted(atom_symbols) + ['Unknown']


def one_hot_encoding(x: Any, allowable_set: Sequence[Any]) -> list[float]:
    """
    Map x to a one-hot vector over allowable_set.

    If x is unseen, we send it to the last slot of the allowable set.
    This keeps the featurizer robust when a rare chemistry case shows up.
    """
    if x not in allowable_set:
        x = allowable_set[-1]
    return [1.0 if x == s else 0.0 for s in allowable_set]


def bool_feature(flag: bool) -> list[float]:
    """Wrap a boolean as a length-1 float list so it can be concatenated with one-hot features."""
    return [float(flag)]


@dataclass
class GraphData:
    """
    Minimal container for one molecule graph.

    x:
        Node feature matrix with shape [num_atoms, num_node_features].
    edge_index:
        Directed edge list with shape [2, num_edges].
    edge_attr:
        Edge feature matrix with shape [num_edges, num_edge_features].
    atom_symbols:
        Human-readable atom labels in RDKit order. This is mainly for inspection.
    """

    smiles: str
    x: torch.Tensor
    edge_index: torch.Tensor
    edge_attr: torch.Tensor
    atom_symbols: list[str]

    @property
    def num_nodes(self) -> int:
        return int(self.x.size(0))

    @property
    def num_edges(self) -> int:
        return int(self.edge_index.size(1))


class SimpleGraphFeaturizer:
    """
    Convert one SMILES string into chemistry-aware graph tensors.

    The design choice here is very deliberate:
    - keep the features local and interpretable
    - include chemistry that the graph encoder would otherwise have to infer indirectly
    - avoid expensive descriptors that would make iteration slower on day one

    These are *input* features only. The Graph Nets layers will transform them into
    learned hidden states later.
    """

    def __init__(
        self,
        atom_types: list[str],
        formal_charge_set: list[Any] | None = None,
        degree_set: list[Any] | None = None,
        hybridization_set: list[Any] | None = None,
        num_h_set: list[Any] | None = None,
        bond_types: list[Any] | None = None,
        bond_stereo_set: list[Any] | None = None,
    ) -> None:
        self.atom_types = atom_types

        # These allowable sets are intentionally small and chemically interpretable.
        # Anything rarer falls into an 'Other' bucket so the featurizer stays robust.
        self.formal_charge_set = formal_charge_set or [-3, -2, -1, 0, 1, 2, 3, 'Other']
        self.degree_set = degree_set or [0, 1, 2, 3, 4, 5, 6, 'Other']
        self.hybridization_set = hybridization_set or [
            Chem.rdchem.HybridizationType.SP,
            Chem.rdchem.HybridizationType.SP2,
            Chem.rdchem.HybridizationType.SP3,
            Chem.rdchem.HybridizationType.SP3D,
            Chem.rdchem.HybridizationType.SP3D2,
            'Other',
        ]
        self.num_h_set = num_h_set or [0, 1, 2, 3, 4, 'Other']

        self.bond_types = bond_types or [
            Chem.rdchem.BondType.SINGLE,
            Chem.rdchem.BondType.DOUBLE,
            Chem.rdchem.BondType.TRIPLE,
            Chem.rdchem.BondType.AROMATIC,
            'Unknown',
        ]
        self.bond_stereo_set = bond_stereo_set or [
            Chem.rdchem.BondStereo.STEREONONE,
            Chem.rdchem.BondStereo.STEREOZ,
            Chem.rdchem.BondStereo.STEREOE,
            Chem.rdchem.BondStereo.STEREOCIS,
            Chem.rdchem.BondStereo.STEREOTRANS,
            Chem.rdchem.BondStereo.STEREOANY,
            'Unknown',
        ]

    @property
    def node_feature_dim(self) -> int:
        """Return the width of one atom feature vector."""
        return (
            len(self.atom_types)
            + len(self.formal_charge_set)
            + len(self.degree_set)
            + len(self.hybridization_set)
            + len(self.num_h_set)
            + 2  # aromatic flag + ring flag
        )

    @property
    def edge_feature_dim(self) -> int:
        """Return the width of one bond feature vector."""
        return len(self.bond_types) + len(self.bond_stereo_set) + 2  # conjugated + ring flags

    def atom_features(self, atom: Chem.rdchem.Atom) -> list[float]:
        """
        Build one atom feature vector.

        Why these features?
        - atom type: basic chemical identity
        - formal charge: ionization often matters for binding and toxicity
        - degree: local connectivity / substitution pattern
        - hybridization: crude local geometry / electronics
        - total hydrogens: helps distinguish local environments like amines vs amides
        - aromatic / ring flags: capture common medicinal-chemistry motifs cheaply
        """
        features: list[float] = []
        features += one_hot_encoding(atom.GetSymbol(), self.atom_types)
        features += one_hot_encoding(atom.GetFormalCharge(), self.formal_charge_set)
        features += one_hot_encoding(atom.GetDegree(), self.degree_set)
        features += one_hot_encoding(atom.GetHybridization(), self.hybridization_set)
        features += one_hot_encoding(atom.GetTotalNumHs(), self.num_h_set)
        features += bool_feature(atom.GetIsAromatic())
        features += bool_feature(atom.IsInRing())
        return features

    def bond_features(self, bond: Chem.rdchem.Bond) -> list[float]:
        """
        Build one directed bond feature vector.

        Why these features?
        - bond type: baseline connectivity chemistry
        - bond stereo: E/Z or cis/trans information when present
        - conjugation: a simple proxy for electron delocalization
        - ring flag: distinguishes chain edges from ring edges
        """
        features: list[float] = []
        features += one_hot_encoding(bond.GetBondType(), self.bond_types)
        features += one_hot_encoding(bond.GetStereo(), self.bond_stereo_set)
        features += bool_feature(bond.GetIsConjugated())
        features += bool_feature(bond.IsInRing())
        return features

    def __call__(self, smiles: str) -> GraphData:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f'Could not parse SMILES: {smiles}')

        atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]

        # RDKit is only used here, at featurization time.
        # After this point the model only sees tensors, not chemistry objects.
        node_features = [self.atom_features(atom) for atom in mol.GetAtoms()]
        x = torch.tensor(node_features, dtype=torch.float)

        edge_indices = []
        edge_features = []
        for bond in mol.GetBonds():
            u = bond.GetBeginAtomIdx()
            v = bond.GetEndAtomIdx()
            bond_feat = self.bond_features(bond)

            # We store both directions explicitly.
            # Message passing usually operates on directed edges, even for undirected molecular graphs.
            edge_indices.append([u, v])
            edge_features.append(bond_feat)
            edge_indices.append([v, u])
            edge_features.append(bond_feat)

        if edge_indices:
            edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
            edge_attr = torch.tensor(edge_features, dtype=torch.float)
        else:
            # Rare edge case: a molecule with one explicit atom and no explicit bonds.
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr = torch.empty((0, self.edge_feature_dim), dtype=torch.float)

        return GraphData(
            smiles=smiles,
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            atom_symbols=atom_symbols,
        )


atom_types = build_atom_vocab(train_smiles)
featurizer = SimpleGraphFeaturizer(atom_types=atom_types)

sample_graph = featurizer(train_smiles[0])
sample_mol = Chem.MolFromSmiles(sample_graph.smiles)
sample_atom = sample_mol.GetAtomWithIdx(0)
sample_bond = sample_mol.GetBondWithIdx(0) if sample_mol.GetNumBonds() > 0 else None

print('example smiles:', sample_graph.smiles)
print('atom symbols:', sample_graph.atom_symbols)
print('x shape:', tuple(sample_graph.x.shape))
print('edge_index shape:', tuple(sample_graph.edge_index.shape))
print('edge_attr shape:', tuple(sample_graph.edge_attr.shape))
print()
print('node feature dimension breakdown:')
print('  atom type one-hot:      ', len(featurizer.atom_types))
print('  formal charge one-hot:  ', len(featurizer.formal_charge_set))
print('  degree one-hot:         ', len(featurizer.degree_set))
print('  hybridization one-hot:  ', len(featurizer.hybridization_set))
print('  total H one-hot:        ', len(featurizer.num_h_set))
print('  aromatic flag:          ', 1)
print('  ring flag:              ', 1)
print('  total node dim:         ', featurizer.node_feature_dim)
print()
print('edge feature dimension breakdown:')
print('  bond type one-hot:      ', len(featurizer.bond_types))
print('  bond stereo one-hot:    ', len(featurizer.bond_stereo_set))
print('  conjugated flag:        ', 1)
print('  ring flag:              ', 1)
print('  total edge dim:         ', featurizer.edge_feature_dim)
print()
print('first atom raw chemistry:')
print({
    'symbol': sample_atom.GetSymbol(),
    'formal_charge': sample_atom.GetFormalCharge(),
    'degree': sample_atom.GetDegree(),
    'hybridization': str(sample_atom.GetHybridization()),
    'total_h': sample_atom.GetTotalNumHs(),
    'is_aromatic': sample_atom.GetIsAromatic(),
    'is_in_ring': sample_atom.IsInRing(),
})
if sample_bond is not None:
    print('first bond raw chemistry:')
    print({
        'bond_type': str(sample_bond.GetBondType()),
        'stereo': str(sample_bond.GetStereo()),
        'is_conjugated': sample_bond.GetIsConjugated(),
        'is_in_ring': sample_bond.IsInRing(),
    })

# Visual inspection is still useful. Graph tensors are abstract; the molecule image helps us
# connect the learned representation back to real chemistry.
display(Draw.MolToImage(sample_mol, size=(300, 300)))


## Dataset And Graph Batch Construction
We batch graphs in a PyG-like way without adding a new dependency:
- concatenate all node tensors across the batch
- concatenate all edge tensors across the batch
- keep a `batch_index` vector that says which graph each node belongs to

Two ideas are worth separating clearly:
- **fixed chemistry features** are computed once by RDKit when each molecule is featurized
- **learned hidden states** are updated many times by the Graph Nets layers during training

This batching scheme only moves tensors around; it does **not** recompute chemistry descriptors inside the model.


In [ ]:
class GraphSelfiesDataset(Dataset):
    """
    Pair one molecule graph with one SELFIES target sequence.

    We featurize on-the-fly instead of storing every graph in memory ahead of time.
    That keeps the notebook simpler and avoids a large memory spike for ChemBL + Zinc.

    In a bigger project we might cache these graph tensors to disk, but for a tutorial-style
    notebook this version keeps the data flow easier to understand.
    """

    def __init__(self, smiles_list: list[str], target_ids: np.ndarray, featurizer: SimpleGraphFeaturizer):
        self.smiles_list = smiles_list
        self.target_ids = target_ids
        self.featurizer = featurizer

    def __len__(self) -> int:
        return len(self.smiles_list)

    def __getitem__(self, idx: int):
        # RDKit featurization happens here, once for this sample retrieval.
        # The graph tensors returned by this method are what the model actually consumes.
        graph = self.featurizer(self.smiles_list[idx])
        return {
            'graph': graph,
            'target_ids': torch.from_numpy(self.target_ids[idx]).long(),
        }


def collate_graph_batch(samples: list[dict]) -> dict[str, Any]:
    """
    Merge a list of single-graph samples into one batched graph.

    Output tensors follow the common graph-learning convention:
    - x:          [total_nodes, node_feature_dim]
    - edge_index: [2, total_edges]
    - edge_attr:  [total_edges, edge_feature_dim]
    - batch:      [total_nodes] with graph ids 0..B-1
    - target_ids: [batch_size, seq_len]

    The `batch` vector is the crucial bookkeeping object: it tells the encoder which nodes
    belong to which molecule when we pool back to one graph representation per sample.
    """
    node_parts = []
    edge_index_parts = []
    edge_attr_parts = []
    batch_parts = []
    target_parts = []
    smiles_batch = []

    node_offset = 0
    for graph_id, sample in enumerate(samples):
        graph = sample['graph']

        node_parts.append(graph.x)
        batch_parts.append(torch.full((graph.num_nodes,), graph_id, dtype=torch.long))
        target_parts.append(sample['target_ids'])
        smiles_batch.append(graph.smiles)

        if graph.num_edges > 0:
            # When we concatenate graphs, every edge index must be shifted so it still points
            # to the correct node rows in the big stacked node matrix.
            edge_index_parts.append(graph.edge_index + node_offset)
            edge_attr_parts.append(graph.edge_attr)

        node_offset += graph.num_nodes

    x = torch.cat(node_parts, dim=0)
    batch_index = torch.cat(batch_parts, dim=0)
    target_ids = torch.stack(target_parts, dim=0)

    if edge_index_parts:
        edge_index = torch.cat(edge_index_parts, dim=1)
        edge_attr = torch.cat(edge_attr_parts, dim=0)
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, featurizer.edge_feature_dim), dtype=torch.float)

    return {
        'x': x,
        'edge_index': edge_index,
        'edge_attr': edge_attr,
        'batch': batch_index,
        'target_ids': target_ids,
        'smiles': smiles_batch,
    }


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_graph_batch)


train_dataset = GraphSelfiesDataset(train_smiles, train_targets, featurizer)
val_dataset = GraphSelfiesDataset(val_smiles, val_targets, featurizer)
test_dataset = GraphSelfiesDataset(test_smiles, test_targets, featurizer)

sample_batch = next(iter(make_loader(train_dataset, batch_size=4, shuffle=True)))
print('batched x shape:', tuple(sample_batch['x'].shape))
print('batched edge_index shape:', tuple(sample_batch['edge_index'].shape))
print('batched edge_attr shape:', tuple(sample_batch['edge_attr'].shape))
print('batched target_ids shape:', tuple(sample_batch['target_ids'].shape))
print('batch vector shape:', tuple(sample_batch['batch'].shape))


## Graph VAE Architecture
The encoder below is a **simplified Graph-Nets-style encoder**:
1. update directed edge states using source node, destination node, and current edge state
2. aggregate incoming edge messages at each node
3. update node states
4. repeat this block three times
5. pool node states into one graph vector and map to `mu` / `logvar`

The decoder intentionally stays close to the old VAE so the comparison stays focused on the encoder change.


In [ ]:
def move_batch_to_device(batch: dict[str, Any], device: torch.device) -> dict[str, Any]:
    """Move the tensor parts of a batch to the selected device and leave metadata alone."""
    moved = {}
    for key, value in batch.items():
        moved[key] = value.to(device) if torch.is_tensor(value) else value
    return moved


def pool_nodes(node_states: torch.Tensor, batch_index: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute sum and mean pooling over nodes that belong to the same graph.

    Using both is a simple way to keep:
    - size-sensitive information (sum)
    - size-normalized information (mean)
    """
    num_graphs = int(batch_index.max().item()) + 1
    hidden_dim = node_states.size(-1)

    sum_pool = node_states.new_zeros((num_graphs, hidden_dim))
    sum_pool.index_add_(0, batch_index, node_states)

    counts = torch.bincount(batch_index, minlength=num_graphs).clamp(min=1).to(node_states.dtype).unsqueeze(-1)
    mean_pool = sum_pool / counts
    return sum_pool, mean_pool


class EdgeNodeMessageBlock(nn.Module):
    """
    One round of Graph-Nets-style message passing.

    We keep this block deliberately explicit:
    - update edge states first
    - aggregate updated edges onto destination nodes
    - update node states from the aggregated messages
    """

    def __init__(self, hidden_dim: int):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.edge_norm = nn.LayerNorm(hidden_dim)
        self.node_norm = nn.LayerNorm(hidden_dim)

    def forward(
        self,
        node_states: torch.Tensor,
        edge_index: torch.Tensor,
        edge_states: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        src, dst = edge_index

        if edge_states.size(0) > 0:
            # Edge update: each directed edge sees the source node, destination node,
            # and its own current edge state.
            edge_inputs = torch.cat([node_states[src], node_states[dst], edge_states], dim=-1)
            edge_delta = self.edge_mlp(edge_inputs)
            edge_states = self.edge_norm(edge_states + edge_delta)

            # Aggregate messages onto the destination node of each directed edge.
            aggregated_messages = node_states.new_zeros((node_states.size(0), edge_states.size(-1)))
            aggregated_messages.index_add_(0, dst, edge_states)

            # Normalize by in-degree so higher-degree nodes do not automatically
            # get larger activations just because they have more neighbors.
            in_degree = node_states.new_zeros((node_states.size(0), 1))
            in_degree.index_add_(0, dst, torch.ones((edge_states.size(0), 1), device=node_states.device))
            aggregated_messages = aggregated_messages / in_degree.clamp(min=1.0)
        else:
            # Safety path for graphs with no explicit bonds.
            aggregated_messages = node_states.new_zeros((node_states.size(0), edge_states.size(-1)))

        # Node update: mix the current node state with the aggregated edge messages.
        node_inputs = torch.cat([node_states, aggregated_messages], dim=-1)
        node_delta = self.node_mlp(node_inputs)
        node_states = self.node_norm(node_states + node_delta)

        return node_states, edge_states


class GraphEncoder(nn.Module):
    """
    Bond-aware graph encoder with three repeated message-passing blocks.
    """

    def __init__(
        self,
        node_feature_dim: int,
        edge_feature_dim: int,
        hidden_dim: int,
        readout_dim: int,
        latent_dim: int,
        num_blocks: int = 3,
    ):
        super().__init__()
        self.node_input_proj = nn.Linear(node_feature_dim, hidden_dim)
        self.edge_input_proj = nn.Linear(edge_feature_dim, hidden_dim)
        self.blocks = nn.ModuleList([EdgeNodeMessageBlock(hidden_dim) for _ in range(num_blocks)])

        # The readout MLP mirrors the old VAE idea of producing one graph vector
        # before the latent heads. We use sum + mean pooling and then project.
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim * 2, readout_dim),
            nn.SiLU(),
        )
        self.mu_head = nn.Linear(readout_dim, latent_dim)
        self.logvar_head = nn.Linear(readout_dim, latent_dim)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
        batch_index: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        node_states = self.node_input_proj(x)

        if edge_attr.size(0) > 0:
            edge_states = self.edge_input_proj(edge_attr)
        else:
            edge_states = x.new_zeros((0, self.node_input_proj.out_features))

        for block in self.blocks:
            node_states, edge_states = block(node_states, edge_index, edge_states)

        sum_pool, mean_pool = pool_nodes(node_states, batch_index)
        graph_state = self.readout(torch.cat([sum_pool, mean_pool], dim=-1))
        return self.mu_head(graph_state), self.logvar_head(graph_state)


class GraphSelfiesVAE(nn.Module):
    """
    Hybrid VAE:
    - graph encoder
    - Gaussian latent bottleneck
    - SELFIES GRU decoder
    """

    def __init__(
        self,
        node_feature_dim: int,
        edge_feature_dim: int,
        vocab_size: int,
        seq_len: int,
        latent_dim: int = LATENT_DIM,
        hidden_dim: int = GRAPH_HIDDEN_DIM,
        readout_dim: int = GRAPH_READOUT_DIM,
        num_blocks: int = GRAPH_BLOCKS,
    ):
        super().__init__()
        self.seq_len = seq_len
        self.vocab_size = vocab_size

        self.encoder = GraphEncoder(
            node_feature_dim=node_feature_dim,
            edge_feature_dim=edge_feature_dim,
            hidden_dim=hidden_dim,
            readout_dim=readout_dim,
            latent_dim=latent_dim,
            num_blocks=num_blocks,
        )

        # Keep the decoder close to the pre-existing VAE for a cleaner comparison.
        self.linear_3 = nn.Linear(latent_dim, DECODER_INPUT_DIM)
        self.gru = nn.GRU(
            input_size=DECODER_INPUT_DIM,
            hidden_size=DECODER_HIDDEN_DIM,
            num_layers=DECODER_LAYERS,
            batch_first=True,
        )
        self.linear_4 = nn.Linear(DECODER_HIDDEN_DIM, vocab_size)

    def sampling(self, mean: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        eps = 1e-2 * torch.randn_like(logvar)
        return torch.exp(0.5 * logvar) * eps + mean

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        z = F.selu(self.linear_3(z))
        z_seq = z.unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.gru(z_seq)
        logits = self.linear_4(out)
        return logits

    def forward(self, batch: dict[str, Any]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mean, logvar = self.encoder(batch['x'], batch['edge_index'], batch['edge_attr'], batch['batch'])
        z = self.sampling(mean, logvar)
        logits = self.decode(z)
        return logits, mean, logvar


def parameter_report(model: nn.Module) -> dict[str, float]:
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb = total_params * 4 / (1024 ** 2)  # float32 parameter memory only
    return {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'parameter_memory_mb': size_mb,
    }


model = GraphSelfiesVAE(
    node_feature_dim=featurizer.node_feature_dim,
    edge_feature_dim=featurizer.edge_feature_dim,
    vocab_size=VOCAB_SIZE,
    seq_len=SEQ_LEN,
).to(device)

sample_batch = move_batch_to_device(sample_batch, device)
with torch.no_grad():
    sample_logits, sample_mean, sample_logvar = model(sample_batch)

report = parameter_report(model)
print('sample logits shape:', tuple(sample_logits.shape))
print('sample mean shape:', tuple(sample_mean.shape))
print('sample logvar shape:', tuple(sample_logvar.shape))
print(f"total parameters: {report['total_params']:,}")
print(f"trainable parameters: {report['trainable_params']:,}")
print(f"parameter memory (float32 only): {report['parameter_memory_mb']:.2f} MB")



## Training Loop
The loop below intentionally mirrors the old VAE training setup:
- token cross-entropy reconstruction loss
- KL term with optional annealing
- token accuracy as an easy sanity metric


In [ ]:
def vae_loss(
    logits: torch.Tensor,
    target_ids: torch.Tensor,
    mean: torch.Tensor,
    logvar: torch.Tensor,
    *,
    pad_idx: int,
    beta: float = 1.0,
    free_bits_nats: float = 0.0,
):
    vocab_size = logits.size(-1)
    recon_sum = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        target_ids.reshape(-1),
        ignore_index=pad_idx,
        reduction='sum',
    )

    kl_per_dim = -0.5 * (1 + logvar - mean.pow(2) - logvar.exp())
    if free_bits_nats > 0:
        kl_per_dim = torch.clamp(kl_per_dim, min=float(free_bits_nats))
    kl_sum = kl_per_dim.sum()

    n_nonpad = (target_ids != pad_idx).sum().clamp(min=1)
    total = recon_sum + beta * kl_sum
    return total, recon_sum, kl_sum, n_nonpad


def kl_beta(epoch: int, anneal_epochs: int) -> float:
    if anneal_epochs <= 1:
        return 1.0
    return float(min(1.0, epoch / anneal_epochs))


def run_epoch(model: nn.Module, loader: DataLoader, *, optimizer=None, beta: float = 1.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_sum = 0.0
    recon_sum = 0.0
    kl_sum = 0.0
    n_samples = 0
    n_nonpad = 0
    n_correct = 0

    for batch in tqdm(loader, leave=False, desc='train' if is_train else 'eval'):
        batch = move_batch_to_device(batch, device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits, mean, logvar = model(batch)
            total, recon, kl, nonpad = vae_loss(
                logits,
                batch['target_ids'],
                mean,
                logvar,
                pad_idx=PAD_IDX,
                beta=beta,
                free_bits_nats=FREE_BITS_NATS,
            )
            if is_train:
                total.backward()
                optimizer.step()

        mask = batch['target_ids'] != PAD_IDX
        preds = logits.argmax(dim=-1)

        total_sum += total.item()
        recon_sum += recon.item()
        kl_sum += kl.item()
        n_samples += batch['target_ids'].size(0)
        n_nonpad += int(nonpad.item())
        n_correct += ((preds == batch['target_ids']) & mask).sum().item()

    return {
        'total': total_sum / max(n_samples, 1),
        'recon_per_token': recon_sum / max(n_nonpad, 1),
        'kl': kl_sum / max(n_samples, 1),
        'token_acc': n_correct / max(n_nonpad, 1),
    }


def evaluate(model: nn.Module, dataset: Dataset, *, beta: float):
    loader = make_loader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    return run_epoch(model, loader, optimizer=None, beta=beta)


def train_model(model: nn.Module, train_dataset: Dataset, val_dataset: Dataset, *, epochs: int = EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    train_loader = make_loader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = make_loader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    history = {
        'beta': [],
        'train_total': [],
        'val_total': [],
        'train_recon_per_token': [],
        'val_recon_per_token': [],
        'train_kl': [],
        'val_kl': [],
        'train_token_acc': [],
        'val_token_acc': [],
    }

    for ep in range(1, epochs + 1):
        beta = kl_beta(ep, KL_ANNEAL_EPOCHS)
        train_metrics = run_epoch(model, train_loader, optimizer=optimizer, beta=beta)
        val_metrics = run_epoch(model, val_loader, optimizer=None, beta=beta)

        history['beta'].append(beta)
        history['train_total'].append(train_metrics['total'])
        history['val_total'].append(val_metrics['total'])
        history['train_recon_per_token'].append(train_metrics['recon_per_token'])
        history['val_recon_per_token'].append(val_metrics['recon_per_token'])
        history['train_kl'].append(train_metrics['kl'])
        history['val_kl'].append(val_metrics['kl'])
        history['train_token_acc'].append(train_metrics['token_acc'])
        history['val_token_acc'].append(val_metrics['token_acc'])

        print(
            f"epoch {ep:03d} | beta={beta:.2f} | "
            f"train total={train_metrics['total']:.4f} | val total={val_metrics['total']:.4f} | "
            f"train token acc={train_metrics['token_acc']:.4f} | val token acc={val_metrics['token_acc']:.4f}"
        )

    return model, history


In [ ]:
if RUN_TRAINING:
    print('Starting graph-VAE training...')
    model, history = train_model(model, train_dataset, val_dataset, epochs=EPOCHS)
    test_metrics = evaluate(model, test_dataset, beta=history['beta'][-1])
    print('Test metrics:', test_metrics)

    plt.figure(figsize=(8, 4))
    plt.plot(history['train_total'], label='train total')
    plt.plot(history['val_total'], label='val total')
    plt.xlabel('epoch')
    plt.ylabel('avg loss per sample')
    plt.title('Graph VAE training history')
    plt.legend()
    plt.show()
else:
    print('RUN_TRAINING is False. Set it to True when you want to launch training.')
